# Source statistics, entropy-bound internals and the accountant
Companion to `paper_tightness.ipynb` (DP-SGD attack + gap analysis) and `paper_utility_curves.ipynb` (realistic
training: bound + utility). This notebook backs every remaining empirical statement of the paper: the dataset table
(App. `app:exp_data`), the reference scale `tab:trsigma` (App. `app:exp_entropy`), the grid / margins / Monte-Carlo
settings of the bound, the Hadamard rotation, the LPIPS map (App. `app:exp_phi`), and the accuracy of the
accountant quadrature (Sec. `sec:bound_gamma`). It only reads result files (plus building small Hadamard factors and
the LPIPS network); run it via `sbatch --export=ALL,NB=notebooks/paper_bound_details.ipynb reproduce/slurm_exec_notebook.sh`.

`CHECK` lines test a statement of the paper against the data and print `TRUE`/`FALSE`; `FALSE` = the **paper text**
must change.

In [1]:
import warnings   # known, benign library warnings (they would print local install paths into the outputs):
warnings.filterwarnings("ignore", message="Optimal order is the largest alpha")   # opacus's PRV accountant sizes its mesh with an RDP bound
warnings.filterwarnings("ignore", message="The parameter 'pretrained' is deprecated")   # inside the lpips package
warnings.filterwarnings("ignore", message="Arguments other than a weight enum")         # inside the lpips package
import glob, json, math, os, sys, re, functools
import numpy as np
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.bound.rho import PIXEL_BOUND_DIR, LPIPS_BOUND_DIR, rho_at_schedule
from src.privacy.gamma import gamma_prv_mean
LN2 = math.log(2.0)
DATASETS = os.environ.get("PAPER_DATASETS", "mnist cifar10 celeba imagenette").split()
NAME = {"mnist": "MNIST", "cifar10": "CIFAR-10", "celeba": "CelebA", "imagenette": "ImageNette"}
SHAPE = {"mnist": (1, 28, 28), "cifar10": (3, 32, 32), "celeba": (3, 64, 64), "imagenette": (3, 128, 128)}
TRS = json.load(open(ROOT / "results/sigma_trace_ci.json"))          # scripts/sigma_trace_ci.py, on D_perp

@functools.lru_cache(maxsize=None)
def bound(ds, rep):
    d = (PIXEL_BOUND_DIR if rep == "pixel" else LPIPS_BOUND_DIR)[ds]
    return json.load(open(Path(d) / "entropy_bound.json"))
def reps_for(ds):      # LPIPS only where Phi is admissible (not ImageNette: Phi's AlexNet was trained on ImageNet)
    return ["pixel"] + (["lpips"] if ds in LPIPS_BOUND_DIR else [])
print("ready")

ready


## Dataset table (App. `app:exp_data`)
C×H×W, d, tr Σ̂ (pixel, grey levels², on D_⊥), RMSE_prior/√d, n_est (the bound's estimation sample) and |D_⊥|
(the sample tr Σ̂ is estimated on; for MNIST/CIFAR-10/CelebA it also fixes the eigenbasis V).

In [2]:
print(f"{'dataset':<11}{'CxHxW':>12}{'d':>8}{'tr Sigma':>12}{'RMSE/sqrt d':>13}{'n_est':>8}{'|D_perp| (trace)':>18}{'n_v (rotation)':>16}{'rotation':>10}")
rows = []
for ds in DATASETS:
    t = TRS[f"pixel|{ds}"]; c = bound(ds, "pixel"); C, H, W = SHAPE[ds]
    assert C * H * W == t["d"] == c["d_raw"]
    rows.append((ds, t, c))
    print(f"{NAME[ds]:<11}{f'{C}x{H}x{W}':>12}{t['d']:>8}{t['trS_hat']:>12.3e}{t['rmse_prior']:>13.1f}{c['n']:>8}"
          f"{t['n']:>18}{c['n_v']:>16}{c['rotation']:>10}")
L = [r"\begin{tabular}{lcccccc}", r"\toprule",
     r"Dataset & $C\!\times\!H\!\times\!W$ & $d$ & $\tr\Sigma$ & $\rmseprior/\sqrt d$ & $n_{\mathrm{est}}$ & $|D_\perp|$ \\", r"\midrule"]
for ds, t, c in rows:
    C, H, W = SHAPE[ds]; m, e = f"{t['trS_hat']:.2e}".split("e")
    L.append(rf"{NAME[ds]} & ${C}\!\times\!{H}\!\times\!{W}$ & ${t['d']:,}$ & ${m}\!\times\!10^{{{int(e)}}}$ & "
             rf"${t['rmse_prior']:.1f}$ & ${c['n']//1000}$k & ${t['n']//1000}$k \\".replace(",", "{,}"))
L += [r"\bottomrule", r"\end{tabular}"]
(ROOT / "tables").mkdir(exist_ok=True); (ROOT / "tables/datasets.tex").write_text("\n".join(L) + "\n"); print("\n".join(L))
for ds, t, c in rows:
    if c["rotation"] != "eig":
        print(f"NOTE {NAME[ds]} pixel bound uses rotation='{c['rotation']}' (U = d^-1/2 H D, random signs), not the "
              f"eigenbasis V of rem:choice_of_U (n_v={c['n_v']}).")

dataset           CxHxW       d    tr Sigma  RMSE/sqrt d   n_est  |D_perp| (trace)  n_v (rotation)  rotation
MNIST           1x28x28     784   3.432e+06         66.2   59000             10000           10000       eig
CIFAR-10        3x32x32    3072   1.244e+07         63.6   49000             10000           10000       eig
CelebA          3x64x64   12288   5.910e+07         69.3   44000            117000          117000       eig
ImageNette    3x128x128   49152   2.457e+08         70.7   10000              1000               0     signs
\begin{tabular}{lcccccc}
\toprule
Dataset & $C\!\times\!H\!\times\!W$ & $d$ & $\tr\Sigma$ & $\rmseprior/\sqrt d$ & $n_{\mathrm{est}}$ & $|D_\perp|$ \\
\midrule
MNIST & $1\!\times\!28\!\times\!28$ & $784$ & $3.43\!\times\!10^{6}$ & $66.2$ & $59$k & $10$k \\
CIFAR-10 & $3\!\times\!32\!\times\!32$ & $3{,}072$ & $1.24\!\times\!10^{7}$ & $63.6$ & $49$k & $10$k \\
CelebA & $3\!\times\!64\!\times\!64$ & $12{,}288$ & $5.91\!\times\!10^{7}$ & $69.3$ & $44$k & 

## Reference scale `tab:trsigma` and its effect on the bound (App. `app:exp_entropy` *Reference scale*)
Point estimate of tr Σ_Ψ on D_⊥, its 95% relative half-width, and RMSE_prior per coordinate. Then: dividing by the
upper end of the interval with the dither scales unchanged leaves the absolute bound (eq. master_abs) unchanged and
rescales ρ by √(tr Σ̂ / tr Σ_UCB), so it lowers a bound ρ* by ρ*·(1 − √(tr Σ̂/tr Σ_UCB)) ≤ 1 − √(tr Σ̂/tr Σ_UCB).

In [3]:
print(f"{'dataset':<11}{'trS px':>12}{'UCB+ px':>9}{'RMSE/sqrt d':>13}{'trS Phi':>10}{'UCB+ Phi':>9}{'RMSE/sqrt m':>13}{'n':>7}")
L = [r"\begin{tabular}{l r r r r r r}", r"\toprule",
     r"& \multicolumn{3}{c}{pixel ($\Psi=\mathrm{id}$)} & \multicolumn{3}{c}{perceptual ($\Psi=\lpipsmap$)} \\",
     r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
     r"Dataset & $\tr\Sigma$ & UCB margin & $\rmseprior/\sqrt d$ & $\tr\Sigma_{\lpipsmap}$ & UCB margin & $\rmseprior/\sqrt m$ \\",
     r"\midrule"]
hws = {"pixel": [], "lpips": []}
for ds in DATASETS:
    p = TRS[f"pixel|{ds}"]; f = TRS[f"lpips|{ds}"] if "lpips" in reps_for(ds) else None   # no LPIPS for ImageNette
    hws["pixel"].append(p["rel_ucb"])
    if f: hws["lpips"].append(f["rel_ucb"])
    print(f"{NAME[ds]:<11}{p['trS_hat']:>12.4e}{100*p['rel_ucb']:>8.2f}%{p['rmse_prior']:>13.2f}"
          + (f"{f['trS_hat']:>10.4f}{100*f['rel_ucb']:>8.2f}%{f['rmse_prior']:>13.3e}" if f else f"{'---':>10}{'---':>9}{'---':>13}")
          + f"{p['n']:>7}")
    mp, ep = f"{p['trS_hat']:.2e}".split("e")
    lp_cells = "--- & --- & ---"
    if f:
        mf, ef = f"{f['rmse_prior']:.2e}".split("e")
        lp_cells = rf"${f['trS_hat']:.4f}$ & ${100*f['rel_ucb']:.1f}\%$ & ${mf}\times10^{{{int(ef)}}}$"
    L.append(rf"{NAME[ds]} & ${mp}\times10^{{{int(ep)}}}$ & ${100*p['rel_ucb']:.1f}\%$ & ${p['rmse_prior']:.1f}$ & " + lp_cells + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]
(ROOT / "tables/trsigma.tex").write_text("\n".join(L) + "\n"); print("\n" + "\n".join(L))
print(f"\nmax relative UCB margin (trS_ucb/trS_hat - 1): pixel {100*max(hws['pixel']):.2f}%   LPIPS {100*max(hws['lpips']):.2f}%")
meths = {TRS[f"{rep}|{ds}"].get("method", "?") for ds in DATASETS for rep in reps_for(ds)}
deltas = {TRS[f"{rep}|{ds}"].get("delta") for ds in DATASETS for rep in reps_for(ds)}
print(f"bound construction (scripts/sigma_trace_ci.py): {meths}, one-sided, delta = {deltas}")
print(f"CHECK every row is a finite-sample empirical-Bernstein bound: "
      f"{'TRUE ' if all('Bernstein' in m for m in meths) else 'FALSE'}")

print("\nlowering of the bound if rho is divided by the upper confidence end of tr Sigma:")
worst = 0.0
for ds in DATASETS:
    for rep in reps_for(ds):
        t = TRS[f"{rep}|{ds}"]; fac = math.sqrt(t["trS_hat"] / t["trS_ucb"])
        rho_max = max(rho_at_schedule(ds, 1e6, 1.0, 1, "mse" if rep == "pixel" else "lpips")["rho"], 0.0)                       # nm -> inf: the largest reported rho*
        worst = max(worst, rho_max * (1 - fac))
        print(f"  {NAME[ds]:<11}{rep:<6} factor sqrt(trS/trS_ucb) = {fac:.4f}   -> rho* lowered by at most {rho_max*(1-fac):.4f}"
              f" ({100*(1-fac):.2f}% relative)")
print(f"max absolute lowering over all reported bounds: {worst:.4f}")

dataset          trS px  UCB+ px  RMSE/sqrt d   trS Phi UCB+ Phi  RMSE/sqrt m      n
MNIST        3.4318e+06    2.47%        66.16    0.1338    5.82%    2.049e-03  10000
CIFAR-10     1.2444e+07    3.19%        63.65    0.2463    3.31%    2.780e-03  10000
CelebA       5.9096e+07    0.46%        69.35    0.1445    0.62%    2.129e-03 117000
ImageNette   2.4569e+08   16.59%        70.70       ---      ---          ---   1000

\begin{tabular}{l r r r r r r}
\toprule
& \multicolumn{3}{c}{pixel ($\Psi=\mathrm{id}$)} & \multicolumn{3}{c}{perceptual ($\Psi=\lpipsmap$)} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-7}
Dataset & $\tr\Sigma$ & UCB margin & $\rmseprior/\sqrt d$ & $\tr\Sigma_{\lpipsmap}$ & UCB margin & $\rmseprior/\sqrt m$ \\
\midrule
MNIST & $3.43\times10^{6}$ & $2.5\%$ & $66.2$ & $0.1338$ & $5.8\%$ & $2.05\times10^{-3}$ \\
CIFAR-10 & $1.24\times10^{7}$ & $3.2\%$ & $63.6$ & $0.2463$ & $3.3\%$ & $2.78\times10^{-3}$ \\
CelebA & $5.91\times10^{7}$ & $0.5\%$ & $69.3$ & $0.1445$ & $0.6\%$ & $2.1

## Grid, confidence budget, Monte-Carlo replicates and realised margins (App. `app:exp_entropy` *Grid*, *Estimator and margins*)
G = K × N per representation, |G|, α and its split, n_MC per block size, and the realised per-dimension margins
t_MD/k and t_MC/k. Margins are reported over the whole grid and at the binding pairs of the privacy-spectrum
configurations (fleet q, T from `results/utility/runs`, at the paper's ε rows).

In [4]:
for ds in DATASETS:
    for rep in reps_for(ds):
        c = bound(ds, rep); P = c["pairs"]
        nmc = sorted({(p["k"], p["n_mc"]) for p in P})
        print(f"{NAME[ds]:<11}{rep:<6} K={c['grid_k']}  N={c['grid_nu2']}  |G|={c['G_size']}  alpha={c['alpha']} "
              f"(MC {c['alpha_mc']}, MD {c['alpha_md']})  n_est={c['n']}  d={c['d']} (raw {c['d_raw']})")
        print(f"{'':17}n_MC per k: {dict(nmc)}")
        tmd = [p["t_md_bits"] / p["k"] for p in P]; tmc = [p["t_mc_bits"] / p["k"] for p in P]
        print(f"{'':17}t_MD = {c['t_md_bits']:.4f} bits (all pairs);  t_MD/k in [{min(tmd):.4f}, {max(tmd):.4f}];  "
              f"t_MC/k in [{min(tmc):.4f}, {max(tmc):.4f}] bits/dim")

# margins at the binding pairs of the spectrum configurations
@functools.lru_cache(maxsize=None)
def fleet_qT(ds):
    model = {"mnist": "lenet5", "cifar10": "wrn16_4_gn", "celeba": "resnet18gn", "imagenette": "resnet18gn"}[ds]
    for f in glob.glob(str(ROOT / "results/utility/runs/*.json")):
        r = json.load(open(f))
        if r.get("dataset") == ds and r.get("model_name") == model and r.get("sampling_rate") and r.get("total_steps"):
            return float(r["sampling_rate"]), int(r["total_steps"])

from src.privacy.accounting import compute_epsilon_prv
def fleet_nms(ds):
    tsv = ROOT / "results/utility/shared_eps_fleet_params.tsv"
    rows = [l.split("\t") for l in open(tsv).read().splitlines() if l.startswith(ds + "\t")]
    return sorted({float(r[3]) for r in rows})
def nm_for_eps(ds, eps):
    """The fleet noise multiplier whose PRV eps (fleet q, T, delta = n^-1.1) is the target eps."""
    q, T = fleet_qT(ds); n = int(round(1024 / q))
    nm = min(fleet_nms(ds), key=lambda s: abs(math.log(compute_epsilon_prv(s, n ** -1.1, q, T) / eps)))
    assert abs(compute_epsilon_prv(nm, n ** -1.1, q, T) / eps - 1) < 0.05
    return nm

print(f"\n{'dataset':<11}{'rep':<6}{'eps':>4}{'k*':>4}{'nu2*':>6}{'t_MD/k':>9}{'t_MC/k':>9}")
for ds in DATASETS:
    q, T = fleet_qT(ds)
    for eps in (1, 4, 16):
        nm = nm_for_eps(ds, eps)
        for rep in reps_for(ds):
            f = rho_at_schedule(ds, nm, q, T, "mse" if rep == "pixel" else "lpips")
            if f["k_star"] is None: continue
            p = next(p for p in bound(ds, rep)["pairs"] if p["k"] == f["k_star"] and abs(p["nu2"] - f["nu2_star"]) < 1e-12)
            print(f"{NAME[ds]:<11}{rep:<6}{eps:>4}{p['k']:>4}{p['nu2']:>6g}{p['t_md_bits']/p['k']:>9.4f}{p['t_mc_bits']/p['k']:>9.4f}")

MNIST      pixel  K=[1, 2, 4, 7, 8, 14, 16, 28]  N=[0.2, 0.4, 0.8, 1.6, 3.2, 6.4]  |G|=48  alpha=0.05 (MC 0.025, MD 0.025)  n_est=59000  d=784 (raw 784)
                 n_MC per k: {1: 2, 2: 4, 4: 4, 7: 7, 8: 8, 14: 14, 16: 16, 28: 28}
                 t_MD = 0.1384 bits (all pairs);  t_MD/k in [0.0049, 0.1384];  t_MC/k in [0.0093, 0.0919] bits/dim
MNIST      lpips  K=[1, 2, 4, 8, 16, 32]  N=[0.2, 0.4, 0.8, 1.6, 3.2, 6.4]  |G|=36  alpha=0.05 (MC 0.025, MD 0.025)  n_est=59000  d=32768 (raw 31872)
                 n_MC per k: {1: 2, 2: 4, 4: 4, 8: 8, 16: 16, 32: 32}
                 t_MD = 0.1357 bits (all pairs);  t_MD/k in [0.0042, 0.1357];  t_MC/k in [0.0012, 0.0123] bits/dim
CIFAR-10   pixel  K=[1, 2, 3, 4, 6, 8, 12, 16, 24, 32]  N=[0.2, 0.4, 0.8, 1.6, 3.2, 6.4]  |G|=60  alpha=0.05 (MC 0.025, MD 0.025)  n_est=49000  d=3072 (raw 3072)
                 n_MC per k: {1: 2, 2: 4, 3: 4, 4: 4, 6: 6, 8: 8, 12: 12, 16: 16, 24: 24, 32: 32}
                 t_MD = 0.1517 bits (all pairs);  t_M


dataset    rep    eps  k*  nu2*   t_MD/k   t_MC/k


MNIST      pixel    1  14   1.6   0.0099   0.0181
MNIST      lpips    1  16   3.2   0.0085   0.0019


MNIST      pixel    4  16   1.6   0.0087   0.0170
MNIST      lpips    4  32   3.2   0.0042   0.0014


MNIST      pixel   16  28   0.8   0.0049   0.0156
MNIST      lpips   16  32   0.8   0.0042   0.0020


...

CelebA     pixel    4   8   0.4   0.0198   0.0119
CelebA     lpips    4  32   3.2   0.0048   0.0016


CelebA     pixel   16  32   0.8   0.0050   0.0048
CelebA     lpips   16  32   0.8   0.0048   0.0023


ImageNette pixel    1  24   3.2   0.0121   0.0040


ImageNette pixel    4  32   3.2   0.0091   0.0035


ImageNette pixel   16  32   0.8   0.0091   0.0050


## The rotation: Hadamard factorisations (App. `app:exp_entropy` *Blocks*, Rem. `rem:choice_of_U`)
The paper states H_784 = H28⊗H28, H_3072 = H12⊗H64⊗H4, H_12288 = H12⊗H64⊗H16, H_49152 = H12⊗H64⊗H64 and, for LPIPS, a zero-pad of
the m = 31,872 features to 32,768 with U = 32768^{-1/2} H_32768 D, H_32768 = H64⊗H64⊗H8 and D a fixed random sign
flip, *"so no eigenbasis is estimated"* (padding fraction 2.7%). Below: the factor orders `src/bound/blocks.hadamard_factors` builds (each
factor's orthogonality F Fᵀ = oI is asserted inside), and the dimension/rotation the bounds actually used.

In [5]:
from src.bound.blocks import hadamard_factors, _order_decomposition, next_hadamard_dim
for d in (784, 3072, 12288, 49152, 31872, 32768):
    try:
        F = hadamard_factors(d); print(f"H_{d:<6} = " + " (x) ".join(f"H{f.shape[0]}" for f in F) + "   (each factor asserted orthogonal)")
    except ValueError as e:
        print(f"H_{d:<6}: no construction ({e.__class__.__name__}); next_hadamard_dim -> {next_hadamard_dim(d)}")
print()
for ds in DATASETS:
    for rep in reps_for(ds):
        c = bound(ds, rep)
        print(f"bound {NAME[ds]:<11}{rep:<6} d_raw={c['d_raw']:>6}  rotated/padded d={c['d']:>6}  rotation={c['rotation']}")
LPDS = [ds for ds in DATASETS if "lpips" in reps_for(ds)]
lp_pad = {bound(ds, "lpips")["d"] for ds in LPDS}
print(f"\nCHECK LPIPS zero-padded from m = 31,872 to 32,768 (padding {(32768 - 31872) / 32768:.1%}): "
      f"{'TRUE ' if lp_pad == {32768} and {bound(ds, 'lpips')['d_raw'] for ds in LPDS} == {31872} else 'FALSE'} (used d = {sorted(lp_pad)})")
print(f"CHECK LPIPS rotation is the random sign flip (no eigenbasis estimated): "
      f"{'TRUE ' if all(bound(ds, 'lpips')['rotation'] == 'signs' for ds in LPDS) else 'FALSE'} "
      f"(rotations = {sorted({bound(ds, 'lpips')['rotation'] for ds in LPDS})})")

H_784    = H28 (x) H28   (each factor asserted orthogonal)
H_3072   = H12 (x) H64 (x) H4   (each factor asserted orthogonal)
H_12288  = H12 (x) H64 (x) H16   (each factor asserted orthogonal)
H_49152  = H12 (x) H64 (x) H64   (each factor asserted orthogonal)
H_31872  = H12 (x) H332 (x) H8   (each factor asserted orthogonal)
H_32768  = H64 (x) H64 (x) H8   (each factor asserted orthogonal)

bound MNIST      pixel  d_raw=   784  rotated/padded d=   784  rotation=eig
bound MNIST      lpips  d_raw= 31872  rotated/padded d= 32768  rotation=signs
bound CIFAR-10   pixel  d_raw=  3072  rotated/padded d=  3072  rotation=eig
bound CIFAR-10   lpips  d_raw= 31872  rotated/padded d= 32768  rotation=signs
bound CelebA     pixel  d_raw= 12288  rotated/padded d= 12288  rotation=eig
bound CelebA     lpips  d_raw= 31872  rotated/padded d= 32768  rotation=signs
bound ImageNette pixel  d_raw= 49152  rotated/padded d= 49152  rotation=signs

CHECK LPIPS zero-padded from m = 31,872 to 32,768 (padding 2.7%): 

## The LPIPS map Φ (App. `app:exp_phi`)
Frozen LPIPS v0.1 with AlexNet backbone, inputs resized to 64×64; m = Σ_ℓ C_ℓ H_ℓ W_ℓ. The layer shapes are read from
the network itself (one probe forward pass in `PhiExtractor`).

In [6]:
from src.bound.lpips_phi import PhiExtractor, LPIPS_IMG_SIZE
ext = PhiExtractor(device="cpu")
for (C, H, W), m in zip(ext.layer_shapes, ext.layer_dims):
    print(f"  layer C={C:>3} H={H:>2} W={W:>2}  -> {m:>6} coords")
print(f"m = {ext.m}   input resize = {LPIPS_IMG_SIZE}x{LPIPS_IMG_SIZE}   backbone = AlexNet (lpips v0.1)")
print(f"CHECK m == 31,872: {'TRUE ' if ext.m == 31872 else 'FALSE'}")

  layer C= 64 H=15 W=15  ->  14400 coords
  layer C=192 H= 7 W= 7  ->   9408 coords
  layer C=384 H= 3 W= 3  ->   3456 coords
  layer C=256 H= 3 W= 3  ->   2304 coords
  layer C=256 H= 3 W= 3  ->   2304 coords
m = 31872   input resize = 64x64   backbone = AlexNet (lpips v0.1)
CHECK m == 31,872: TRUE 


## Accountant quadrature (Sec. `sec:bound_gamma`: *"each K_t^PRV is a one-dimensional integral, evaluated to an error
negligible relative to the finite-sample margins"*)
`gamma_prv_mean` uses a fixed 256-node Gauss–Hermite rule. Compared here with adaptive `scipy.integrate.quad`
(tolerance 1e-13) at every (q, σ, T) the paper evaluates: the spectrum fleet (4 datasets × 11 ε levels) and the attack
(single step q=1 is closed-form; subsampled q=0.02, T=100). The error in γ enters the bound as an extra
|Δγ|/k bits per dimension, compared against the smallest entropy margin (t_MD + t_MC)/k anywhere in the grids.

In [7]:
from scipy.integrate import quad
def kl_quad(q, s):                                           # KL(P||Q) nats, P=(1-q)N(0,s^2)+qN(1,s^2), Q=N(0,s^2)
    if q >= 1: return 0.5 / s**2
    lr = lambda x: np.logaddexp(math.log1p(-q), math.log(q) + (2 * x - 1) / (2 * s * s))
    pdf = lambda x, mu: math.exp(-(x - mu)**2 / (2 * s * s)) / math.sqrt(2 * math.pi * s * s)
    f = lambda x: ((1 - q) * pdf(x, 0.0) + q * pdf(x, 1.0)) * lr(x)
    lo, hi = -40 * s - 1, 40 * s + 2
    return quad(f, lo, hi, epsabs=1e-15, epsrel=1e-13, limit=500, points=[0.0, 0.5, 1.0])[0]

cfgs = []
for ds in DATASETS:
    q, T = fleet_qT(ds)
    for nm in fleet_nms(ds): cfgs.append((f"fleet {ds}", q, T, nm))
from src.attacks.spm import solve_sigma
spm = json.load(open(ROOT / "results/tightness_spm/spm_mnist_pixel_base.json"))    # the attack's subsampled schedule
for g in [r["gamma"] for r in spm["single"]]: cfgs.append(("attack subsampled", spm["q"], spm["T"], solve_sigma(spm["T"], spm["q"], g)))

worst_rel, worst_abs_bits = 0.0, 0.0
for lab, q, T, s in cfgs:
    gh = gamma_prv_mean(T, q, s) / LN2; ex = T * kl_quad(q, s) / LN2
    rel = abs(gh - ex) / ex; worst_rel = max(worst_rel, rel); worst_abs_bits = max(worst_abs_bits, abs(gh - ex))
print(f"{len(cfgs)} configurations; max relative error of the 256-node rule = {worst_rel:.2e}; max |Delta gamma| = {worst_abs_bits:.2e} bits")
min_margin = min((p["t_md_bits"] + p["t_mc_bits"]) / p["k"] for ds in DATASETS for rep in reps_for(ds) for p in bound(ds, rep)["pairs"])
print(f"smallest entropy margin (t_MD + t_MC)/k over all grids = {min_margin:.4f} bits/dim; "
      f"largest quadrature effect |Delta gamma|/k <= {worst_abs_bits:.2e} bits/dim (k >= 1)")
print(f"CHECK quadrature error negligible vs the margins (ratio < 1e-3): {'TRUE ' if worst_abs_bits / min_margin < 1e-3 else 'FALSE'}"
      f"  (ratio {worst_abs_bits/min_margin:.1e})")

50 configurations; max relative error of the 256-node rule = 3.74e-10; max |Delta gamma| = 6.67e-09 bits
smallest entropy margin (t_MD + t_MC)/k over all grids = 0.0054 bits/dim; largest quadrature effect |Delta gamma|/k <= 6.67e-09 bits/dim (k >= 1)
CHECK quadrature error negligible vs the margins (ratio < 1e-3): TRUE   (ratio 1.2e-06)
